In [1]:
import numpy as np
import rasterio
from pathlib import Path
import geopandas as gpd
from rasterio.mask import mask
import json
from datetime import date
import calendar

# =========================================================
# PATHS
# =========================================================
ROOT = Path.cwd().parent
quarter_dir  = ROOT / "data" / "quarterly_tiffs"
threshold_dir = ROOT / "data" / "percentile_tiffs"
output_dir   = ROOT / "data" / "monthly_heatday_scores"
geojson_path = ROOT.parent / "Maps" / "Geojson" / "assam_district_2024-11.geojson"
output_dir.mkdir(parents=True, exist_ok=True)

# =========================================================
# LOAD DISTRICT GEOMETRY
# =========================================================
gdf  = gpd.read_file(geojson_path)
geoms = json.loads(gdf.to_json())["features"]
geoms = [f["geometry"] for f in geoms]

# =========================================================
# LOAD SINGLE BAND RASTER
# =========================================================
def load_raster(path):
    with rasterio.open(path) as src:
        return src.read(1)

# =========================================================
# QUARTER → MONTH BAND OFFSETS
# =========================================================
def get_band_slice_for_month(year: int, month: int):
    """
    Return (band_start, band_end) — 1-based, inclusive — for the
    given month inside its quarterly TIF.

    ERA-5 Land quarterly TIFs are assumed to be stored as one band
    per day, ordered Jan-1 … Dec-31 within the quarter.

    Q1 → months 1-3  (Jan / Feb / Mar)
    Q2 → months 4-6  (Apr / May / Jun)
    Q3 → months 7-9  (Jul / Aug / Sep)
    Q4 → months 10-12 (Oct / Nov / Dec)
    """
    quarter_first_month = ((month - 1) // 3) * 3 + 1   # 1, 4, 7, or 10

    # Count days before this month within the quarter
    day_offset = 0
    for m in range(quarter_first_month, month):
        day_offset += calendar.monthrange(year, m)[1]

    days_in_month = calendar.monthrange(year, month)[1]

    band_start = day_offset + 1                  # 1-based
    band_end   = day_offset + days_in_month      # inclusive
    return band_start, band_end


# =========================================================
# MONTHLY LOOP
# =========================================================

years = [2021, 2022, 2023, 2024, 2025, 2026]

# ---------------------------------------------------------
# Determine the latest available quarter for the last year
# ---------------------------------------------------------
last_year = max(years)

quarter_to_months = {
    "Q1": (1, 3),
    "Q2": (4, 6),
    "Q3": (7, 9),
    "Q4": (10, 12),
}

available_quarters = []

for q in ["Q1", "Q2", "Q3", "Q4"]:
    hi_file = quarter_dir / f"HI_{last_year}_{q}.tif"
    if hi_file.exists():
        available_quarters.append(q)

if len(available_quarters) == 0:
    raise FileNotFoundError(
        f"No quarterly HI files found for {last_year}."
    )

last_available_quarter = available_quarters[-1]
last_month = quarter_to_months[last_available_quarter][1]

print(f"\nLatest available quarter for {last_year}: {last_available_quarter}")

# ---------------------------------------------------------
# Process years
# ---------------------------------------------------------
for year in years:

    print(f"\n================ YEAR {year} ================")

    # Earlier years: all months
    # Latest year: only up to available quarter
    if year == last_year:
        months = range(1, last_month + 1)
    else:
        months = range(1, 13)

    for month in months:

        print(f"\nProcessing {year}-{month:02d}")

        # -------------------------------------------------
        # SEASON / QUARTER MAPPING
        # -------------------------------------------------
        if month in [1, 2, 3]:
            season, q = "JFM", "Q1"
        elif month in [4, 5, 6]:
            season, q = "AMJ", "Q2"
        elif month in [7, 8, 9]:
            season, q = "JAS", "Q3"
        else:
            season, q = "OND", "Q4"

        hi_path = quarter_dir / f"HI_{year}_{q}.tif"

        if not hi_path.exists():
            print("SKIP: HI missing")
            continue

        # -------------------------------------------------
        # LOAD THRESHOLDS
        # -------------------------------------------------
        try:
            p80 = load_raster(threshold_dir / f"{season}_P80_1990_2023.tif")
            p88 = load_raster(threshold_dir / f"{season}_P88_1990_2023.tif")
            p95 = load_raster(threshold_dir / f"{season}_P95_1990_2023.tif")
            p99 = load_raster(threshold_dir / f"{season}_P99_1990_2023.tif")
        except Exception as e:
            print(f"SKIP: missing thresholds for {season} — {e}")
            continue

        # -------------------------------------------------
        # FIGURE OUT WHICH BANDS BELONG TO THIS MONTH
        # -------------------------------------------------
        band_start, band_end = get_band_slice_for_month(year, month)
        expected_days = calendar.monthrange(year, month)[1]

        print(
            f"  Quarter {q}, bands {band_start}–{band_end} "
            f"({expected_days} days)"
        )

        # -------------------------------------------------
        # LOAD ONLY THIS MONTH'S BANDS FROM THE QUARTERLY TIF
        # -------------------------------------------------
        with rasterio.open(hi_path) as src:

            total_bands = src.count

            if band_end > total_bands:
                print(
                    f"  WARN: TIF has only {total_bands} bands. "
                    f"Using available bands."
                )
                band_end = total_bands

            if band_start > total_bands:
                print("  SKIP: No data available for this month.")
                continue

            band_indices = list(range(band_start, band_end + 1))

            hi = src.read(band_indices)
            meta = src.meta.copy()

        days, rows, cols = hi.shape
        print(f"  Loaded {days} daily layers")

        # -------------------------------------------------
        # MONTHLY HEATDAY SCORE
        # -------------------------------------------------
        monthly = np.zeros((rows, cols), dtype=np.float32)

        for d in range(days):

            h = hi[d]

            score = np.zeros((rows, cols), dtype=np.uint8)

            score[(h >= p80) & (h < p88)] = 1
            score[(h >= p88) & (h < p95)] = 2
            score[(h >= p95) & (h < p99)] = 3
            score[h >= p99] = 4

            monthly += score

        # =========================================================
        # WRITE TEMP RASTER
        # =========================================================
        tmp_file = output_dir / f"_tmp_{year}_{month:02d}.tif"

        NODATA_VAL = -9999.0

        meta.update(
            {
                "count": 1,
                "dtype": "float32",
                "nodata": NODATA_VAL,
            }
        )

        with rasterio.open(tmp_file, "w", **meta) as dst:
            dst.write(monthly, 1)

        # =========================================================
        # CLIP TO DISTRICT BOUNDARIES
        # =========================================================
        with rasterio.open(tmp_file) as src:

            out_image, out_transform = mask(
                src,
                geoms,
                crop=True,
                filled=True,
                nodata=NODATA_VAL,
                all_touched=True,
            )

        out_image = out_image.astype("float32")

        out_final = out_image[0].copy()
        out_final[out_final == NODATA_VAL] = np.nan

        out_meta = meta.copy()
        out_meta.update(
            {
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform,
                "count": 1,
                "nodata": np.nan,
            }
        )

        # =========================================================
        # FINAL SAVE
        # =========================================================
        final_file = output_dir / f"HEATDAY_{year}_{month:02d}.tif"

        with rasterio.open(final_file, "w", **out_meta) as dst:
            dst.write(out_final, 1)

        tmp_file.unlink()

        print(
            f"  Saved: {final_file} "
            f"[valid={int(np.sum(~np.isnan(out_final)))} px, "
            f"max_score={np.nanmax(out_final) if np.any(~np.isnan(out_final)) else 'N/A'}]"
        )


Latest available quarter for 2026: Q1

================ YEAR 2021 ================

Processing 2021-01
  Quarter Q1, bands 1–31 (31 days)
  Loaded 31 daily layers
  Saved: /Users/saurabhlevin/Deployment/IDS-DRR_Heat/Heat-assam/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2021_01.tif [valid=1265 px, max_score=15.0]

Processing 2021-02
  Quarter Q1, bands 32–59 (28 days)
  Loaded 28 daily layers


  Saved: /Users/saurabhlevin/Deployment/IDS-DRR_Heat/Heat-assam/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2021_02.tif [valid=1265 px, max_score=14.0]

Processing 2021-03
  Quarter Q1, bands 60–90 (31 days)
  Loaded 31 daily layers
  Saved: /Users/saurabhlevin/Deployment/IDS-DRR_Heat/Heat-assam/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2021_03.tif [valid=1265 px, max_score=64.0]

Processing 2021-04
  Quarter Q2, bands 1–30 (30 days)
  Loaded 30 daily layers


  Saved: /Users/saurabhlevin/Deployment/IDS-DRR_Heat/Heat-assam/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2021_04.tif [valid=1265 px, max_score=13.0]

Processing 2021-05
  Quarter Q2, bands 31–61 (31 days)
  Loaded 31 daily layers
  Saved: /Users/saurabhlevin/Deployment/IDS-DRR_Heat/Heat-assam/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2021_05.tif [valid=1265 px, max_score=26.0]

Processing 2021-06
  Quarter Q2, bands 62–91 (30 days)
  Loaded 30 daily layers


  Saved: /Users/saurabhlevin/Deployment/IDS-DRR_Heat/Heat-assam/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2021_06.tif [valid=1265 px, max_score=45.0]

Processing 2021-07
  Quarter Q3, bands 1–31 (31 days)
  Loaded 31 daily layers
  Saved: /Users/saurabhlevin/Deployment/IDS-DRR_Heat/Heat-assam/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2021_07.tif [valid=1265 px, max_score=31.0]

Processing 2021-08
  Quarter Q3, bands 32–62 (31 days)
  Loaded 31 daily layers


  Saved: /Users/saurabhlevin/Deployment/IDS-DRR_Heat/Heat-assam/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2021_08.tif [valid=1265 px, max_score=20.0]

Processing 2021-09
  Quarter Q3, bands 63–92 (30 days)
  Loaded 30 daily layers
  Saved: /Users/saurabhlevin/Deployment/IDS-DRR_Heat/Heat-assam/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2021_09.tif [valid=1265 px, max_score=32.0]

Processing 2021-10
  Quarter Q4, bands 1–31 (31 days)
  Loaded 31 daily layers


...

  Saved: /Users/saurabhlevin/Deployment/IDS-DRR_Heat/Heat-assam/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2025_06.tif [valid=1265 px, max_score=70.0]

Processing 2025-07
  Quarter Q3, bands 1–31 (31 days)
  Loaded 31 daily layers
  Saved: /Users/saurabhlevin/Deployment/IDS-DRR_Heat/Heat-assam/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2025_07.tif [valid=1265 px, max_score=50.0]

Processing 2025-08
  Quarter Q3, bands 32–62 (31 days)
  Loaded 31 daily layers


  Saved: /Users/saurabhlevin/Deployment/IDS-DRR_Heat/Heat-assam/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2025_08.tif [valid=1265 px, max_score=18.0]

Processing 2025-09
  Quarter Q3, bands 63–92 (30 days)
  Loaded 30 daily layers
  Saved: /Users/saurabhlevin/Deployment/IDS-DRR_Heat/Heat-assam/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2025_09.tif [valid=1265 px, max_score=37.0]

Processing 2025-10
  Quarter Q4, bands 1–31 (31 days)
  Loaded 31 daily layers
  Saved: /Users/saurabhlevin/Deployment/IDS-DRR_Heat/Heat-assam/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2025_10.tif [valid=1265 px, max_score=68.0]

Processing 2025-11


  Quarter Q4, bands 32–61 (30 days)
  Loaded 30 daily layers
  Saved: /Users/saurabhlevin/Deployment/IDS-DRR_Heat/Heat-assam/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2025_11.tif [valid=1265 px, max_score=8.0]

Processing 2025-12
  Quarter Q4, bands 62–92 (31 days)
  Loaded 31 daily layers


  Saved: /Users/saurabhlevin/Deployment/IDS-DRR_Heat/Heat-assam/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2025_12.tif [valid=1265 px, max_score=9.0]

================ YEAR 2026 ================

Processing 2026-01
  Quarter Q1, bands 1–31 (31 days)
  Loaded 31 daily layers
  Saved: /Users/saurabhlevin/Deployment/IDS-DRR_Heat/Heat-assam/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2026_01.tif [valid=1265 px, max_score=11.0]

Processing 2026-02
  Quarter Q1, bands 32–59 (28 days)
  Loaded 28 daily layers


  Saved: /Users/saurabhlevin/Deployment/IDS-DRR_Heat/Heat-assam/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2026_02.tif [valid=1265 px, max_score=30.0]

Processing 2026-03
  Quarter Q1, bands 60–90 (31 days)
  Loaded 31 daily layers
  Saved: /Users/saurabhlevin/Deployment/IDS-DRR_Heat/Heat-assam/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2026_03.tif [valid=1265 px, max_score=42.0]
